In [ ]:
import torch
from sklearn.decomposition import PCA
#from torch_pca import PCA
from tqdm import tqdm
import numpy as np

In [ ]:
from train import prepare_dataloader
from mmfi_dataset.decode_config import MMFIConfig
config_model =  MMFIConfig.load("/visinf/home/mb_tweige/master-thesis-csi-flow/dataset_configs/seemo_config_E8_time.yml")
config_model.modalities = ["csi"]
#config_model.train.batch_size = 1

train_loader, val_loader,test_loader = prepare_dataloader(config_model,num_workers=5,shuffle_val=False) # has to be zero because it does not render properly with multiple workers

len(train_loader.dataset), len(val_loader.dataset),len(test_loader.dataset)

In [ ]:
from matplotlib import pyplot as plt
%env CUDA_VISIBLE_DEVICES=1
device = "cuda"
pca = PCA(svd_solver="full")

X = []
COUNT = 3000
for X1,X2 in tqdm(train_loader, total=COUNT):
    if not COUNT:
        break
    X.append(X1["csi"])
    COUNT -= 1
    
csi = torch.concat(X,dim=0)


In [ ]:
def prepare_tensor(torch_tensor)->np.ndarray:
    N, A,C,T = torch_tensor.shape
    tensor_flat_np = torch_tensor.permute(0,3,1,2).flatten(start_dim=2).flatten(0,1).numpy()
    return tensor_flat_np

In [ ]:
N, A,C,T = csi.shape
tensor = torch.concat([csi.abs(),csi.angle()], dim=1)
tensor_flat_np = prepare_tensor(tensor)
print(tensor_flat_np.shape)

pca.fit(tensor_flat_np)
pca.n_components_

In [ ]:
csi_sample = tensor[0:1,:,:,:]
plt.imshow(csi_sample[0,0], aspect='auto')
plt.show()
print(csi_sample.shape)
tensor_np = prepare_tensor(csi_sample)
transformed =pca.transform(tensor_np)
denoised = pca.inverse_transform(transformed)
print(denoised.shape)
denoised = torch.Tensor(denoised.reshape(1,T,2*A,C)).permute(0,2,3,1).cpu()
plt.imshow(denoised[0,0,:,:], aspect='auto')
plt.show()


In [ ]:
comps = torch.Tensor(pca.components_)
torch.save(comps, "pca_components.pt")

# Use the saved components to transform and reconstruct the sample
transformed_manual = tensor_np @ comps.T.numpy()
denoised_manual = transformed_manual @ comps.numpy()
denoised_manual_reshaped = torch.Tensor(denoised_manual.reshape(1, T, 2*A, C)).permute(0, 2, 3, 1)

# Visualize the result
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(csi_sample[0, 0].cpu(), aspect='auto')
axes[0].set_title('Original CSI')
axes[1].imshow(denoised_manual_reshaped[0, 0].cpu(), aspect='auto')
axes[1].set_title('Denoised CSI (using saved components)')
plt.tight_layout()
plt.show()

In [ ]:
from WIFlow.csi_preprocessor import CSIPreprocessor


class PCAPreprocessor(CSIPreprocessor):
    def __init__(self, n_antenna, n_subcarrier, saved_components_path="pca_components.pt", preserved_components:int=None):
        super().__init__( n_antenna, n_subcarrier,)

        self.components = torch.load(saved_components_path)
        if preserved_components:
            self.components = self.components[:,:preserved_components]
        self.output_dim = n_antenna * 2

    def forward(self, csi: torch.Tensor) -> torch.Tensor:
        N, A, C, T = csi.shape
        tensor = super().forward(csi)
        print(tensor.shape)
        tensor_flat = tensor.permute(0,3,1,2).flatten(start_dim=2).flatten(0,1)
        transformed = tensor_flat @ self.components
        denoised_flat = transformed @ self.components.T
        
        denoised = denoised_flat.reshape(1, T, 2*A, C).permute(0, 2, 3, 1)
        return denoised
for preserved_comp in [0, 3000, 2100, 1500, 1000, 500, 200,100, 10,1]:
    pca_pro = PCAPreprocessor(n_antenna=4, n_subcarrier=30, saved_components_path="pca_components.pt",preserved_components=preserved_comp)
    denoised_manual = pca_pro(csi[:1])
    # Visualize the result
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(csi_sample[0, 0].cpu(), aspect='auto')
    axes[0].set_title(f'Original CSI {preserved_comp} components')
    axes[1].imshow(denoised_manual[0, 0].cpu(), aspect='auto')
    axes[1].set_title('Denoised CSI (using saved components)')
    plt.tight_layout()
    plt.show()

In [ ]:
COUNT = 10
for X1,X2 in test_loader:
    if not COUNT:
        break

    csi = X1["csi"].to(device)
    N, A,C,T = csi.shape
    csi_tensor = torch.concat([csi.abs(),csi.angle()], dim=1)

    tensor = torch.concat([csi.abs(),csi.angle()], dim=1)
    tensor_flatten = tensor.permute(0,3,1,2).flatten(start_dim=1)
    print(tensor_flatten.shape)
    #iterate through samples in batch, because PCA does not support batch processing

    transformed = pca.transform(tensor_flatten)
    denoised_csi_flattened = pca.inverse_transform(transformed)
    denoised = denoised_csi_flattened.reshape(N, T, 2*A, C).permute(0, 2, 3, 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(csi_tensor[0, 0, :, :].cpu(), aspect='auto')
    axes[0].set_title('Original CSI with ')
    axes[1].imshow(denoised[0, 0, :, :].cpu(), aspect='auto')
    axes[1].set_title('Denoised CSI')
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    axes[1].set_xticks([])
    axes[1].set_yticks([])
    plt.tight_layout()
    plt.show()
    COUNT-=1
